In [ ]:
import pandas as pd
import numpy as np
import faiss
import time
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from openai import OpenAI

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)


In [ ]:
FILE_NAME = "leetcode_dataset - lc.csv"

# ── 1. Load ──────────────────────────────────────────────────────────────────
df = pd.read_csv(FILE_NAME)


In [ ]:


# ── 2. Topic filtering ───────────────────────────────────────────────────────
DISCARD_TOPICS = {
    'Database', 'Shell', 'Concurrency',
    'Interactive', 'Brainteaser', 'Geometry',
    'Game Theory', 'Probability and Statistics'
}

KEEP_TOPICS = {
    'Array', 'String', 'Hash Table', 'Math', 'Dynamic Programming',
    'Sorting', 'Greedy', 'Depth-First Search', 'Binary Search',
    'Bit Manipulation', 'Matrix', 'Tree', 'Breadth-First Search',
    'Two Pointers', 'Prefix Sum', 'Heap (Priority Queue)', 'Simulation',
    'Counting', 'Graph Theory', 'Binary Tree', 'Stack', 'Sliding Window',
    'Enumeration', 'Design', 'Backtracking', 'Union-Find', 'Number Theory',
    'Linked List', 'Ordered Set', 'Segment Tree', 'Monotonic Stack',
    'Divide and Conquer', 'Combinatorics', 'Trie', 'Bitmask', 'Queue',
    'Recursion', 'Binary Indexed Tree', 'Memoization', 'Hash Function',
    'Binary Search Tree', 'Topological Sort', 'Shortest Path',
    'String Matching', 'Rolling Hash', 'Data Stream', 'Monotonic Queue',
    'Doubly-Linked List', 'Merge Sort', 'Randomized', 'Counting Sort',
    'Quickselect', 'Suffix Array', 'Sweep Line', 'Minimum Spanning Tree',
    'Bucket Sort', 'Reservoir Sampling', 'Eulerian Circuit', 'Radix Sort',
    'Strongly Connected Component', 'Rejection Sampling', 'Biconnected Component'
}

def parse_topics(topics_str):
    if not isinstance(topics_str, str):
        return set()
    return {t.strip() for t in topics_str.split(',')}

def is_relevant(topics_str):
    topics = parse_topics(topics_str)
    if topics & DISCARD_TOPICS:
        return False
    return bool(topics & KEEP_TOPICS)


df = df[df['related_topics'].apply(is_relevant)].reset_index(drop=True)


In [ ]:

# ── 3. Clean HTML from descriptions ──────────────────────────────────────────
def clean_html(text):
    if isinstance(text, str) and '<' in text:
        return BeautifulSoup(text, "html.parser").get_text(separator=" ").strip()
    return text

df['description_clean'] = df['description'].apply(clean_html)


In [ ]:

# ── 4. Drop short descriptions ────────────────────────────────────────────────
df = df[df['description_clean'].str.len() >= 100].reset_index(drop=True)

print(f"Final dataset shape: {df.shape}")


In [ ]:
# df

In [ ]:

# ── 5. Embed ──────────────────────────────────────────────────────────────────
client = OpenAI(api_key="YOUR_API_KEY")

def get_embeddings_batch(texts, batch_size=100):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(
            input=batch,
            model="text-embedding-3-small"
        )
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"Embedded {min(i+batch_size, len(texts))}/{len(texts)}")
        time.sleep(0.5)
    return np.array(all_embeddings, dtype='float32')

embeddings = get_embeddings_batch(df['description_clean'].tolist())
print(f"Embeddings shape: {embeddings.shape}")


In [ ]:

# ── 6. Build and save FAISS index ─────────────────────────────────────────────
index = faiss.IndexFlatL2(1536)
index.add(embeddings)
faiss.write_index(index, "leetcode.index")

# ── 7. Save cleaned dataframe ─────────────────────────────────────────────────
df.to_csv("leetcode_clean.csv", index=False)

print("Done. leetcode.index and leetcode_clean.csv saved.")